In [18]:
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier

import nltk
# Download NLTK stopwords (only first time)
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.sequence import pad_sequences

import string
import re

import torch
import torch.nn as nn


from pathlib import Path


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
data_path = r'C:\Users\Krish\Downloads\Streamlit_projects\Streamlit_projects\Projects\Spam_Classifier\data\combined_data.csv'

data  = pd.read_csv(data_path)

print(data.sample(4))
print(data.shape)

       label                                               text
11367      1  dear sir ,\ni am dr james alabi , the chairman...
75998      0  wednesday may escapenumber escapenumber watch ...
70036      0  on escapenumber escapenumber escapenumber jona...
80063      1  death is a dialogue between the spirit and the...
(83448, 2)


In [30]:
test = data.sample(10000)

In [31]:
X = data["text"]
y = data['label']

In [ ]:

def rem_stopwords(text):
    stp_words = set(stopwords.words("english"))

    return " ".join(
        word for word in text.split() if word not in stp_words
    )

def rem_punctuation(text):
    return "".join(
        char for char in text if char not in string.punctuation
    )



def remove_tags(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # Remove Markdown links: [text](url)
    text = re.sub(r'\[.*?\]\(.*?\)', ' ', text)

    # Remove HTML/XML tags: <tag>...</tag>
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove escaped characters: \n, \t, \r
    text = re.sub(r'\\[ntr]', ' ', text)

    # Remove remaining special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text.lower()

def tokenize(text):
   return word_tokenize(text)
     
    
def create_vocab(tokens):
    vocab = {
        word: idx + 1
        for idx, word in enumerate(sorted(set(tokens)))
    }

    return vocab



def padding(sequences, max_len=315):
    return pad_sequences(
        sequences,
        maxlen=max_len,
        padding="post",
        truncating="post"
    )

def encode(tokens, vocab):

    return [
        vocab.get(word, 0)
        for word in tokens
    ]

In [37]:
def preprocess(text, vocab):

    ## Remove stopwords
    text_pr = rem_stopwords(text)

    ## Remove punctuation
    text_pr = rem_punctuation(text_pr)

    ## Remove tags
    text_pr = remove_tags(text_pr)

    ## Tokenize
    text_pr = tokenize(text_pr)

    ## Convert words → IDs
    sequence = encode(text_pr, vocab)

    ## Padding
    padded = padding([sequence], max_len=315)

    ## Embedding
    input_ids = torch.tensor(
        padded,
        dtype=torch.long
    )

    embedding_dim = 128

    embedding = nn.Embedding(
        num_embeddings=len(vocab) + 1,
        embedding_dim=embedding_dim
    )

    embedded = embedding(input_ids)

    ## Mean Pooling
    X = embedded.detach().mean(dim=1).numpy()

    return X

In [ ]:
all_words = set()
X_t = X.apply(tokenize)

for tokens in X_t :
    all_words.update(tokens)

vocab = {
    word: idx + 1
    for idx, word in enumerate(sorted(all_words))
}

In [ ]:
X_tr = preprocess(X, vocab) 
model = RandomForestClassifier()
model.fit(X_tr, y)

TypeError: preprocess() missing 1 required positional argument: 'vocab'